# xLSTM-GRPO · Colab A100 (Astral)

수정된 **관측 스키마 v2 / 순자산 변화 보상 / 체결 시뮬레이터 / 날짜별 검증**을 사용합니다.
실제 체결량, 수수료, 세금, 지연, 부분 체결 및 미청산 수량을 반영하며 과거의 매수·승리 보너스는 사용하지 않습니다.
`seq_len`과 `episode_steps`는 **틱 수**이고, `max_holding_seconds`는 실제 경과 초입니다.

먼저 수정된 프로젝트 코드를 GitHub에 반영한 뒤 `REVISION`을 지정하거나, 수정본 폴더를 `/content/stock-bot2`에 준비하세요.
이 노트북만 업로드하고 이전 학습 코드를 clone하면 변경 사항이 적용되지 않습니다.
재추출한 `data/extracted_episodes_v2` 폴더를 Drive의 `ColabData/datasets/stockbot/20260811/extracted_episodes_v2`에 업로드한 뒤 실행하세요.
모델은 새로운 `scalping_v3_a100_astral_return_priority_run01` 실험 폴더에 저장합니다.
기존 `grpo_xlstm_v4` 체크포인트를 사용하려면 현재 관측 스키마와 날짜 이력이 있어야 하며, `MODE`와 `LOAD_POLICY`를 명시해야 합니다.
관측 스키마와 날짜 이력이 없는 구형 체크포인트는 재사용할 수 없습니다.

순서대로 실행하세요. 실제 A100에서의 속도·메모리는 **GPU 사전 점검 셀**이 측정합니다.
설정은 학습 시작점이며 수익성이나 최적 하이퍼파라미터를 의미하지 않습니다.

기본값은 **1단계 매수·전량 매도**(`MAX_STAGES=1`)입니다. 필요하면 설정 셀에서 최대 5단계까지 분할 매매를 설정할 수 있습니다.
시장 특징 27개에 단계별 활성 상태·손익률·보유시간 비율 3개씩을 더해 관측 차원은 **42개**입니다.
부분 체결된 수량과 현금을 기준으로 포지션을 갱신하고, 매도는 FIFO로 정산합니다. 날짜 분할은 train/validation/test를 사용합니다.

새 추출 파일은 `schema_version=2`, 원본 특징 27개, 원화 가격과 실제 호가·수량 배열을 사용합니다.
기존 백만원 단위 데이터로 학습한 모델과 관측 스키마가 달라지므로 기본 `MODE='new'`로 시작하세요.

분할 수는 `MAX_STAGES`(1~5, 기본 1)로 설정합니다. `1`은 분할 없이 매수·전량 매도합니다. 관측값은 사용하지 않는 슬롯을 0으로 채워 42차원을 유지합니다. `resume`/`finetune`에서는 체크포인트의 분할 수를 복원하므로, 다른 분할 수로 학습하려면 `MODE='new'`와 새 `RUN_NAME`을 사용하세요.

수익률 우선 기본 설정: `large` 모델(CNN 256 / xLSTM 512 / FC 512), 100만 학습 스텝, 시간 할인 없는 `gamma=1.0`, 검증 64회입니다. 수수료·세금·체결 비용을 포함한 검증 순수익률로 체크포인트를 선택합니다. 메모리가 부족하면 미니배치·worker 수를 줄이며 모델 폭은 자동 축소하지 않습니다. 이 설정이 최적 수익률을 보장하지는 않습니다. 기존 모델과 같은 검증 조건에서 비교하고, 테스트는 최종 확인에만 사용하세요. 새 설정은 `MODE='new'`에서 적용되며, `resume`/`finetune`은 이전 모델·학습 설정을 복원합니다.

미청산 물량이 있는 검증 후보는 최고 모델 선택에서 제외합니다. 전량 청산 조건을 만족하는 후보가 없으면 학습은 최종 모델을 확정하지 않고 오류를 보고하며, `checkpoint_iter*.pt`는 진단용으로 보존합니다. 양의 순수익률과 실제 체결 여부를 함께 확인하세요.


## 1. A100 및 시스템 RAM 확인

런타임에서 A100을 선택하세요. [A100은 40GB/80GB 모델이 있으며](https://www.nvidia.com/en-us/data-center/a100/),
[Colab의 GPU 및 시스템 RAM 할당은 세션에 따라 달라집니다](https://research.google.com/colaboratory/faq.html).
GPU 종류와 사용 가능한 RAM을 실제로 확인하여 설정합니다.

In [ ]:
import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import torch

GiB = 1024 ** 3

def available_ram_gib():
    entries = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
    return int(entries['MemAvailable'].split()[0]) * 1024 / GiB

if not torch.cuda.is_available() or 'A100' not in torch.cuda.get_device_name(0):
    raise RuntimeError('런타임 → 런타임 유형 변경에서 A100 GPU를 선택하세요.')
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / GiB
if VRAM_GIB < 30:
    raise RuntimeError('이 프로필은 전체 A100용입니다. 작은 MIG 파티션은 별도 설정이 필요합니다.')
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
print(f'{GPU_NAME} | VRAM {VRAM_GIB:.1f} GiB | 여유 RAM {available_ram_gib():.1f} GiB | CPU {CPU_COUNT}')
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')

## 2. Drive 연결 및 학습 의존성 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Colab에 설치된 CUDA용 PyTorch를 유지합니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'duckdb>=1.4', 'pandas>=2.0', 'pyarrow>=14', 'python-dotenv>=1.0',
                'gymnasium>=0.29', 'scikit-learn>=1.4', 'tensorboard>=2.14'], check=True)

## 3. 수정된 프로젝트 코드 준비

Private 저장소는 Colab Secret의 `GITHUB_TOKEN`을 사용합니다. 토큰은 원격 URL에 저장하지 않습니다.
이미 준비된 폴더는 그대로 사용합니다. 업데이트가 필요하면 수정본을 반영한 `REVISION`을 지정하세요.

In [ ]:
from google.colab import userdata
import tempfile

REPO_PATH = Path('/content/stock-bot2')
REPO_URL = 'https://github.com/gblue1223/stock-bot2.git'
REVISION = ''  # @param {type:"string"}

try:
    github_token = userdata.get('GITHUB_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    github_token = None

git_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
try:
    with tempfile.TemporaryDirectory() as temporary:
        if github_token:
            askpass = Path(temporary) / 'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\n'
                               'print("x-access-token" if "username" in sys.argv[1].lower() '
                               'else os.environ["COLAB_GIT_TOKEN"])\n')
            askpass.chmod(0o700)
            git_env.update(GIT_ASKPASS=str(askpass), COLAB_GIT_TOKEN=github_token)
        if not REPO_PATH.exists():
            subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], env=git_env, check=True)
        if REVISION.strip() and not (REPO_PATH / '.git').exists():
            raise RuntimeError('직접 업로드한 폴더에는 REVISION을 비우세요. Git checkout에서만 revision을 변경합니다.')
        if REVISION.strip():
            dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH, text=True)
            if dirty.strip():
                raise RuntimeError('수정된 파일이 있어 revision을 바꾸지 않았습니다. 수정본을 먼저 보존하세요.')
            subprocess.run(['git', 'fetch', 'origin', REVISION.strip()], cwd=REPO_PATH, env=git_env, check=True)
            subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_PATH, check=True)
finally:
    git_env.pop('COLAB_GIT_TOKEN', None)
    github_token = None

required = ['lib/observations.py', 'ai_trader/grpo/environments/execution.py',
            'ai_trader/grpo/evaluation.py', 'ai_trader/grpo/backtest.py']
if any(not (REPO_PATH / name).is_file() for name in required):
    raise RuntimeError('이전 프로젝트 코드입니다. 리뷰 수정본이 포함된 REVISION 또는 폴더를 준비하세요.')
os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
from ai_trader.grpo.train_xlstm import TrainingConfig, create_environment, prepare_date_splits
from lib.observations import SCHEMA_VERSION, ObservationBuilder
if SCHEMA_VERSION != 2 or not hasattr(TrainingConfig(), 'execution_config'):
    raise RuntimeError('노트북과 프로젝트 버전이 맞지 않습니다. 수정본으로 런타임을 다시 준비하세요.')
if (REPO_PATH / '.git').exists():
    CODE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_PATH, text=True).strip()
else:
    CODE_REVISION = 'uploaded-local-snapshot'
source_digest = hashlib.sha256()
for source in sorted([*(REPO_PATH / 'ai_trader/grpo').rglob('*.py'), *(REPO_PATH / 'lib').rglob('*.py')]):
    source_digest.update(source.relative_to(REPO_PATH).as_posix().encode())
    source_digest.update(source.read_bytes())
SOURCE_SHA256 = source_digest.hexdigest()
print('사용 코드:', CODE_REVISION)

## 4. 원시 NPZ 데이터 복사

Drive의 추출 데이터 경로를 지정하세요. manifest와 NPZ만 사용하며, 재실행해도 폴더가 중첩되지 않습니다.
기존 27개 특징 데이터의 명시된 레거시 가격 단위는 로더가 변환합니다.
실제 매수·매도 호가와 수량 배열이 없는 데이터는 합성 스프레드 체결로 표시됩니다.
실제 호가가 포함된 새 추출 데이터는 아래 `REQUIRE_ORDER_BOOK=True`로 검사할 수 있습니다.

폴더 이름이 `20260811`이어도 manifest에 **서로 다른 거래일이 최소 3개** 있어야 날짜별 train/validation/test 분할이 가능합니다. 한 거래일만 있다면 여러 거래일을 포함한 추출 데이터 경로를 지정하세요.

현재 기본 데이터는 실제 10단계 호가·수량을 포함하므로 `REQUIRE_ORDER_BOOK=True`로 검사합니다. 원본에 호가 갱신 시각은 없어, 갱신 지연에 따른 호가 신선도는 검증할 수 없습니다.

In [ ]:
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/ColabData/datasets/stockbot/20260811/extracted_episodes_v2')
manifest_bytes = (DRIVE_DATA_DIR / 'manifest.json').read_bytes()
MANIFEST_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
manifest = json.loads(manifest_bytes)
if not manifest.get('episodes'):
    raise ValueError('manifest에 에피소드가 없습니다.')
LOCAL_DATA_DIR = Path('/content') / f'extracted_episodes_{MANIFEST_SHA256[:12]}'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

copies = []
for entry in manifest['episodes']:
    relative = Path(entry['file_path'])
    source = (DRIVE_DATA_DIR / relative).resolve()
    target = (LOCAL_DATA_DIR / relative).resolve()
    if not source.is_relative_to(DRIVE_DATA_DIR.resolve()) or not target.is_relative_to(LOCAL_DATA_DIR.resolve()):
        raise ValueError('manifest의 에피소드 경로가 데이터 폴더를 벗어납니다.')
    source_stat = source.stat()
    if (not target.exists() or target.stat().st_size != source_stat.st_size
            or abs(target.stat().st_mtime - source_stat.st_mtime) > 1):
        copies.append((source, target, source_stat.st_size))
needed_bytes = sum(size for _, _, size in copies)
if shutil.disk_usage('/content').free < needed_bytes + 2 * GiB:
    raise RuntimeError(f'로컬 디스크 공간 부족: 복사 {needed_bytes / GiB:.1f} GiB + 여유 2 GiB 필요')
for index, (source, target, _) in enumerate(copies, 1):
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    if index % 500 == 0:
        print(f'복사 {index}/{len(copies)}')
(LOCAL_DATA_DIR / 'manifest.json').write_bytes(manifest_bytes)
print(f'에피소드 {len(manifest["episodes"]):,}개, 복사 {len(copies):,}개 → {LOCAL_DATA_DIR}')

## 5. A100 프로필 및 실험 설정

| 항목 | A100 40GB | A100 80GB |
|---|---:|---:|
| 관측 틱 / 에피소드 틱 | 1,024 / 300 | 1,024 / 300 |
| CNN / mLSTM / FC 차원 | 256 / 512 / 512 | 256 / 512 / 512 |
| 학습 미니배치 시작값 | 16 | 32 |
| 회당 에피소드 / 그룹 | 16 / 4 | 16 / 4 |
| gradient checkpoint 분할 / update epoch | 16 / 2 | 16 / 2 |
| worker 상한 | 4 | 8 |

GPU 여유 메모리·CPU·RAM에 따라 축소합니다. 두 GPU에서 관측·모델 크기는 유지합니다.
FP32 rollout 한 벌은 약 **0.77 GiB**이며 결합 배열, worker 캐시 및 프로세스 메모리가 추가로 필요합니다.
여유 RAM이 16 GiB보다 작으면 회당 8개 에피소드를 사용합니다.

`MODE='new'`가 기본입니다. `resume`은 저장된 관측·학습·체결 설정과 optimizer/진행률을 복원합니다.
`finetune`은 호환되는 설정과 가중치를 읽고 optimizer/진행률을 새로 시작합니다.
`TOTAL_TIMESTEPS`는 재개 시 **누적 목표치**이므로 저장된 값보다 크게 지정하세요.
새 학습/미세조정에는 비어 있는 `RUN_NAME`을 사용하세요.

수수료·세금과 지연·스프레드 값은 예시 시뮬레이션 가정입니다. 사용하는 시장/계좌/데이터에 맞춰 조정하세요.

In [ ]:
def model_obs_dim(config):
    from lib.observations import ACCOUNT_FIELDS, EXECUTION_FIELDS
    return (config['features'] + 15
            + (len(ACCOUNT_FIELDS) if config.get('account_observations', False) else 0)
            + (len(EXECUTION_FIELDS) if config.get('execution_observations', False) else 0))

def estimated_host_gib(config):
    states = (config['episodes_per_group'] * config['num_groups'] * config['episode_steps']
              * config['seq_len'] * model_obs_dim(config) * 4 / GiB)
    # 원본+concat+임시 배열에 3배, worker RSS/cache 및 부모/eval 여유를 더한 추정치.
    extra_eval_workers = max(0, config.get('evaluation_workers', 1) - 1)
    # Validation environments share one process/cache; count their path arrays only.
    eval_arrays = (extra_eval_workers * 4 * (config['seq_len'] + config['episode_steps']
                   + config.get('liquidation_max_steps', 0)) * model_obs_dim(config) * 4 / GiB)
    return 3 * states + config['num_workers'] * (0.75 + config['cache_max_bytes'] / GiB) + 2 + eval_arrays

def a100_profile(vram_gib, free_vram_gib, ram_gib, cpu_count):
    if vram_gib < 30 or free_vram_gib < 4 or ram_gib < 6:
        raise ValueError('A100 프로필을 시작할 여유 GPU/RAM이 부족합니다.')
    large = vram_gib >= 70
    worker_cap = 8 if large else 4
    ram_workers = 8 if ram_gib >= 48 else (4 if ram_gib >= 24 else 2)
    profile = dict(seq_len=1024, features=27, episode_steps=300,
                   account_observations=False, liquidation_max_steps=0, evaluation_workers=1,
                   cnn_channels=256, rnn_hidden_dim=512, hidden_dim=512,
                   batch_size=32 if large else 16, checkpoint_segments=16,
                   episodes_per_group=4 if ram_gib >= 16 else 2, num_groups=4,
                   num_workers=max(1, min(worker_cap, ram_workers, max(1, cpu_count - 1))),
                   cache_max_bytes=128 * 1024 ** 2)
    if free_vram_gib < 16:
        profile['batch_size'] = min(profile['batch_size'], 8)
    while estimated_host_gib(profile) > 0.70 * ram_gib:
        if profile['num_workers'] > 1:
            profile['num_workers'] //= 2
        elif profile['episodes_per_group'] > 2:
            profile['episodes_per_group'] = 2
        else:
            raise ValueError('롤아웃 저장 공간 부족: 다른 메모리 사용을 줄이거나 고용량 RAM 런타임을 선택하세요.')
    return profile

def restore_run_config(base, checkpoint, mode):
    if mode not in ('resume', 'finetune'):
        raise ValueError('mode must be resume or finetune')
    ObservationBuilder.from_schema(checkpoint.get('observation_schema'))
    saved = checkpoint.get('extra_state', {}).get('training_config')
    if not saved or not checkpoint.get('extra_state', {}).get('date_splits'):
        raise ValueError('학습 설정/날짜 이력이 없는 구형 체크포인트입니다. 신규 학습하세요.')
    merged = dict(saved)
    merged.setdefault('max_stages', checkpoint['observation_schema']['max_stages'])
    merged.setdefault('selection_require_liquidation', True)
    merged['account_observations'] = checkpoint['observation_schema']['version'] in (3, 4)
    merged['execution_observations'] = checkpoint['observation_schema']['version'] == 4
    merged.setdefault('decision_interval_seconds', 0.0)
    merged.setdefault('episode_duration_seconds', 0.0)
    merged.setdefault('group_advantage_coef', 1.0)
    merged.setdefault('training_seed', base.get('training_seed', 42))
    merged.setdefault('liquidation_max_steps', 0)
    merged.setdefault('lambda_gae', 0.95)
    for key in ('evaluation_workers', 'diagnostics_interval', 'diagnostics_max_samples'):
        merged[key] = base[key]
    # 세션 자원과 파일 위치만 새 값 사용. 거래/날짜/관측/optimizer 관련 설정은 보존.
    for key in ('extracted_dir', 'output_dir', 'device', 'load_policy', 'total_timesteps',
                'num_workers', 'batch_size', 'cache_max_bytes', 'checkpoint_segments'):
        merged[key] = base[key]
    merged['resume'] = mode == 'resume'
    if merged['resume'] and not all(key in checkpoint for key in
            ('optimizer_state_dict', 'total_timesteps', 'num_updates', 'iteration')):
        raise ValueError('resume에는 optimizer와 진행률이 포함된 전체 학습 체크포인트가 필요합니다.')
    if merged['resume'] and merged['total_timesteps'] <= checkpoint['total_timesteps']:
        raise ValueError('TOTAL_TIMESTEPS를 체크포인트의 누적 timesteps보다 크게 지정하세요.')
    if mode == 'finetune':
        merged['lr'] = base['lr']
    return merged

In [ ]:
MODE = 'new'  # @param ["new", "resume", "finetune"]
RUN_NAME = 'scalping_v3_a100_astral_return_priority_run01'  # @param {type:"string"}
LOAD_POLICY = ''  # @param {type:"string"}
TOTAL_TIMESTEPS = 1_000_000  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
MAX_STAGES = 1  # @param {type:"integer"}
REQUIRE_ORDER_BOOK = True  # @param {type:"boolean"}
ENABLE_TF32 = True  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}

if MODE not in ('new', 'resume', 'finetune') or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('', '.', '..'):
    raise ValueError('MODE와 단일 폴더 이름 RUN_NAME을 확인하세요.')
OUTPUT_DIR = Path('/content/drive/MyDrive/ColabData/stockbot/models') / RUN_NAME
if MODE != 'resume' and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError('새 학습/미세조정은 비어 있는 RUN_NAME을 지정하세요.')
profile = a100_profile(VRAM_GIB, torch.cuda.mem_get_info()[0] / GiB, available_ram_gib(), CPU_COUNT)
CONFIG = dict(TrainingConfig().__dict__)
CONFIG.update(profile)
CONFIG.update(extracted_dir=str(LOCAL_DATA_DIR), output_dir=str(OUTPUT_DIR), device='cuda',
              total_timesteps=TOTAL_TIMESTEPS, lr=LEARNING_RATE, gamma=1.0, clip=0.1, kl_target=0.01,
              entropy_coef=0.01, value_coef=0.5, max_grad_norm=0.5, num_epochs=2, use_gae=True,
              rolling_window_size=512, rolling_min_samples=64, use_raw_data=True,
              evaluation_episodes=64, evaluation_interval=5, evaluation_seed=SEED, training_seed=SEED,
              account_observations=False, execution_observations=False,
              decision_interval_seconds=0.0, episode_duration_seconds=0.0, group_advantage_coef=1.0, liquidation_max_steps=0, evaluation_workers=1,
              selection_require_liquidation=True,
              validation_fraction=0.2, test_fraction=0.2, embargo_dates=0,
              train_end_date=None, validation_end_date=None, checkpoint_interval=1,
              initial_cash=1_000_000.0, max_stages=MAX_STAGES, max_holding_seconds=300.0, stop_loss_pct=2.0,
              transaction_cost_rate=0.00015, buy_tax_rate=0.0, sell_tax_rate=0.0018,
              no_trade_penalty=0.0, win_bonus=0.0, loss_penalty=0.0, buy_signal_bonus=0.0,
              step_reward_scale=1.0, max_trades_per_episode=None, revert_patience=0,
              load_policy=LOAD_POLICY or None, resume=False,
              execution_config=dict(order_latency_ms=100, cancel_latency_ms=50,
                                    order_ttl_seconds=2.0, spread_bps=10, slippage_bps=2,
                                    fallback_depth=100, require_order_book=REQUIRE_ORDER_BOOK,
                                    max_quote_age_seconds=1.0, tick_size=0.0))
loaded_checkpoint = None
if MODE == 'new' and LOAD_POLICY:
    raise ValueError('LOAD_POLICY를 비우거나 MODE를 resume/finetune으로 바꾸세요.')
if MODE != 'new':
    if not LOAD_POLICY or not Path(LOAD_POLICY).is_file():
        raise FileNotFoundError('재개/미세조정할 체크포인트 경로를 지정하세요.')
    if (MODE == 'resume' and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir())
            and not Path(LOAD_POLICY).resolve().is_relative_to(OUTPUT_DIR.resolve())):
        raise ValueError('다른 실험의 출력에 덮어쓸 수 없습니다. 해당 폴더의 체크포인트 또는 새 RUN_NAME을 사용하세요.')
    # 직접 생성한 신뢰할 수 있는 체크포인트만 사용하세요.
    loaded_checkpoint = torch.load(LOAD_POLICY, map_location='cpu', weights_only=True)
    CONFIG = restore_run_config(CONFIG, loaded_checkpoint, MODE)
    if MODE == 'resume':
        prior_info = Path(LOAD_POLICY).parent.parent / 'colab_run.json'
        if prior_info.exists():
            prior = json.loads(prior_info.read_text())
            if prior['manifest_sha256'] != MANIFEST_SHA256:
                raise ValueError('재개할 실험의 데이터 manifest가 달라졌습니다.')
            ENABLE_TF32 = prior['enable_tf32']
            SEED = prior['seed']
    print('체크포인트 설정 복원 완료. finetune의 학습률만 현재 LR 설정을 사용합니다.')

host_estimate = estimated_host_gib(CONFIG)
if host_estimate > 0.70 * available_ram_gib():
    raise MemoryError(f'저장된 롤아웃 설정은 현재 RAM 예산을 넘습니다: 추정 {host_estimate:.1f} GiB')
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print(f'호스트 학습 메모리 추정 {host_estimate:.1f} GiB (상한 보장 아님)')

## 6. 날짜 분리 및 관측/체결 사전 검사

날짜를 대략 60/20/20으로 나누며 같은 날짜를 여러 분할에서 사용하지 않습니다.
각 분할의 관측 길이를 확인하고 **훈련 분할만** 실제로 열어 관측·체결 모델을 검사합니다.
validation으로 best를 선택하고, 학습 종료 후 선택된 모델을 test에서 평가합니다.

In [ ]:
import numpy as np
from ai_trader.grpo.evaluation import normalize_date, validate_checkpoint_dates

unknown = set(CONFIG) - set(TrainingConfig().__dict__)
if unknown:
    raise ValueError(f'현재 trainer에서 지원하지 않는 설정: {unknown}')
training_config = TrainingConfig()
for key, value in CONFIG.items():
    setattr(training_config, key, value)
training_config.validate()
DATE_SPLITS = prepare_date_splits(training_config)
for partition, dates in DATE_SPLITS.items():
    lengths = [int(ep['length']) for ep in manifest['episodes'] if normalize_date(ep['date']) in dates]
    full = sum(length >= CONFIG['seq_len'] + CONFIG['episode_steps'] for length in lengths)
    if not full:
        raise ValueError(f'{partition}: 관측+학습 틱 수를 만족하는 에피소드가 없습니다.')
    print(f'{partition}: {dates[0]} ~ {dates[-1]}, {len(dates)}일, 충분한 길이 {full}/{len(lengths)}개')
if loaded_checkpoint is not None:
    validate_checkpoint_dates(loaded_checkpoint, DATE_SPLITS)
probe_env = create_environment(training_config, 'cpu', DATE_SPLITS['train'])
try:
    probe_observation, probe_info = probe_env.reset(seed=SEED)
    if loaded_checkpoint is not None:
        probe_env.observation_builder.validate_schema(loaded_checkpoint['observation_schema'])
    assert probe_observation.shape == (CONFIG['seq_len'], model_obs_dim(CONFIG))
    assert np.isfinite(probe_observation).all()
    print('관측:', probe_observation.shape, '| 가격 단위:', probe_env.price_unit)
    print('체결 모델:', probe_info.get('execution_model', probe_env.simulator.summary()['execution_model']))
    OBSERVATION_SCHEMA = probe_env.observation_schema
finally:
    probe_env.close()
    type(probe_env).clear_episode_cache()
    del probe_env
loaded_checkpoint = None
gc.collect()

## 7. GPU 배치 사전 점검

현재 모델의 forward/backward와 Adam 갱신을 실행해 메모리를 측정합니다.
OOM 또는 GPU 여유 공간의 70%를 초과하면 미니배치를 절반으로 줄입니다.
이 검사는 전체 데이터 학습의 최대 메모리를 보장하지 않으므로 실제 학습 로그도 확인하세요.

[TF32](https://docs.pytorch.org/docs/stable/notes/cuda.html)는 FP32 텐서를 유지하면서 내부 행렬 연산 정밀도를 낮춥니다.
동일 설정을 **실제 학습 자식 프로세스에도 적용**합니다. 현재 mLSTM의 AMP/BF16 학습은 사용하지 않습니다.

In [ ]:
from ai_trader.grpo.policies.scalping_policy_xlstm import GRPOPolicyE2EXLSTM

# 학습 프로세스에서 재사용할 코드. PyTorch 2.9+와 이전 설정 API를 섞지 않습니다.
GPU_SETUP = """
import os, random, numpy as np, torch
enable_tf32 = os.environ.get('SCALPING_TF32', '1') == '1'
if hasattr(torch.backends.cuda.matmul, 'fp32_precision'):
    torch.backends.fp32_precision = 'ieee'
    torch.backends.cuda.matmul.fp32_precision = 'tf32' if enable_tf32 else 'ieee'
    torch.backends.cudnn.conv.fp32_precision = 'tf32' if enable_tf32 else 'ieee'
else:
    torch.backends.cuda.matmul.allow_tf32 = enable_tf32
    torch.backends.cudnn.allow_tf32 = enable_tf32
torch.backends.cudnn.benchmark = False
torch.set_num_threads(1)
seed = int(os.environ.get('SCALPING_SEED', '42'))
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
"""
os.environ['SCALPING_TF32'] = '1' if ENABLE_TF32 else '0'
os.environ['SCALPING_SEED'] = str(SEED)
exec(GPU_SETUP)

def probe_batch(config, observation):
    policy = GRPOPolicyE2EXLSTM(obs_dim=model_obs_dim(config),
        cnn_channels=config['cnn_channels'], rnn_hidden_dim=config['rnn_hidden_dim'],
        fc_hidden_dim=config['hidden_dim'], max_stages=config['max_stages'], checkpoint_segments=config['checkpoint_segments']).cuda()
    if config['load_policy']:
        checkpoint = torch.load(config['load_policy'], map_location='cpu', weights_only=True)
        policy.load_state_dict(checkpoint['policy_state_dict'], strict=True)
        del checkpoint
    optimizer = torch.optim.Adam(policy.parameters(), lr=config['lr'])
    batch = torch.as_tensor(observation, device='cuda').unsqueeze(0).repeat(config['batch_size'], 1, 1)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for _ in range(2):
        policy.eval()
        with torch.no_grad():
            actions, _ = policy.get_action(batch)
        policy.train()
        log_probs, entropy, values = policy.evaluate_actions(batch, actions)
        loss = -log_probs.mean() - config['entropy_coef'] * entropy.mean() + config['value_coef'] * values.square().mean()
        if not torch.isfinite(loss):
            raise FloatingPointError('사전 점검 loss에 NaN/Inf가 있습니다.')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), config['max_grad_norm'], error_if_nonfinite=True)
        optimizer.step()
    torch.cuda.synchronize()
    return {'batch_size': config['batch_size'], 'peak_gib': torch.cuda.max_memory_allocated() / GiB,
            'two_steps_seconds': time.perf_counter() - started, 'grad_norm': float(norm)}

GPU_PROBE = None
while CONFIG['batch_size'] >= 8:
    gc.collect()
    torch.cuda.empty_cache()
    memory_budget = 0.70 * torch.cuda.mem_get_info()[0] / GiB
    try:
        result = probe_batch(CONFIG, probe_observation)
        print(result)
        if result['peak_gib'] <= memory_budget:
            GPU_PROBE = result
            break
    except torch.cuda.OutOfMemoryError:
        print(f'배치 {CONFIG["batch_size"]}: GPU 메모리 부족')
    CONFIG['batch_size'] //= 2
gc.collect()
torch.cuda.empty_cache()
if GPU_PROBE is None:
    raise RuntimeError('사전 점검 실패. GPU 메모리를 비우고 관측/모델 설정을 확인하세요.')
PROBED_CONFIG = json.dumps(CONFIG, sort_keys=True)
print('확정 미니배치:', CONFIG['batch_size'])

## 8. 학습 시작

기본 목표는 실제 수집 스텝 1,000,000개입니다. 학습 시간보다 비용 반영 검증 수익률을 우선하며, 검증은 64개 경로에서 평가합니다. 학습 예산 소진이 수익성 확보를 뜻하지는 않습니다.
회당 체크포인트를 Drive에 저장합니다. 중단된 학습은 `checkpoint_iter<N>.pt`를 명시하여 재개하세요.
`checkpoint_best.pt`는 validation에서 선택된 모델입니다. 평가 간격은 5회이며 마지막 회도 평가합니다.
재개 시 출발 모델과 호환되는 기존 best를 현재 validation에서 재평가하여 더 나은 모델을 보존합니다.
`resume`은 optimizer/누적 진행률을 이어가지만 난수 상태까지 동일한 비트 단위 재현을 보장하지 않습니다.

In [ ]:
import signal

if GPU_PROBE is None or PROBED_CONFIG != json.dumps(CONFIG, sort_keys=True):
    raise RuntimeError('현재 CONFIG로 GPU 사전 점검을 먼저 실행하세요.')
if MODE != 'resume' and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError('이미 실행한 실험입니다. 새 RUN_NAME 또는 명시적인 resume으로 설정 셀부터 실행하세요.')
if estimated_host_gib(CONFIG) > 0.70 * available_ram_gib():
    raise MemoryError('학습 직전 여유 RAM이 부족합니다.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_DIR / 'colab_config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
run_info = dict(gpu=GPU_NAME, vram_gib=VRAM_GIB, available_ram_gib=available_ram_gib(),
                torch_version=str(torch.__version__), cuda_version=torch.version.cuda,
                code_revision=CODE_REVISION, source_sha256=SOURCE_SHA256, manifest_sha256=MANIFEST_SHA256,
                mode=MODE, enable_tf32=ENABLE_TF32, seed=SEED, gpu_probe=GPU_PROBE,
                observation_schema=OBSERVATION_SCHEMA)
(OUTPUT_DIR / 'colab_run.json').write_text(json.dumps(run_info, indent=2, ensure_ascii=False), encoding='utf-8')
child_env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1',
                 PYTHONUNBUFFERED='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
                 SCALPING_TF32='1' if ENABLE_TF32 else '0', SCALPING_SEED=str(SEED))
runner = GPU_SETUP + "\nfrom ai_trader.grpo.train_xlstm import main\nraise SystemExit(0 if main() else 1)\n"
command = [sys.executable, '-u', '-c', runner, '--config', str(CONFIG_PATH)]
log_path = OUTPUT_DIR / f'colab_train_{time.strftime("%Y%m%d_%H%M%S")}.log'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(command, cwd=REPO_PATH, env=child_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', bufsize=1,
                               start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        raise
    finally:
        process.stdout.close()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print('학습 완료:', OUTPUT_DIR)

## 9. TensorBoard 및 비용 포함 평가 결과

`validation/mean_net_return`을 중심으로 KL, 거래 수, 보유 시간을 확인하세요.
아래 수익률은 비용 포함 순자산 기준이며 미청산 포지션이 있으면 확정 실현 손익과 다릅니다.
`synthetic_execution_episodes`가 양수이면 실제 호가 재현이 아닌 합성 체결이 포함되었습니다.
test 결과로 하이퍼파라미터를 반복 선택하면 test도 검증 데이터가 되므로 새 홀드아웃이 필요합니다.

In [ ]:
from IPython.display import display
import pandas as pd
from tensorboard import notebook as tb_notebook

tb_notebook.start('--logdir ' + str(OUTPUT_DIR / 'tensorboard_logs'))
report_path = OUTPUT_DIR / 'evaluation_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    fields = ['mean_net_return', 'mean_realized_net_pnl', 'mean_num_trades', 'max_drawdown',
              'avg_holding_time', 'incomplete_liquidation_episodes', 'max_open_quantity',
              'synthetic_execution_episodes', 'num_episodes']
    display(pd.DataFrame({part: {key: report[part].get(key) for key in fields}
                          for part in ('validation', 'test')}))
    print('체결 모델:', {part: report[part]['execution_models'] for part in ('validation', 'test')})
else:
    print('평가 보고서는 학습 완료 후 생성됩니다. 중단된 경우 저장 체크포인트에서 재개하세요.')

## 10. 체결 가정 스트레스 검사 (선택)

`RUN_STRESS_TEST=True`로 실행합니다. **validation**에만 지연 250ms / 합성 스프레드 20bps /
슬리피지 5bps를 적용합니다. 실제 호가가 있으면 스프레드는 해당 호가를 따르므로 `spread_bps`는 합성 체결에만 적용됩니다.
비용 변화에 따른 수익률과 미청산 수량을 확인하세요. 최종 test 보고서는 학습 셀에서 생성한 것을 유지합니다.

In [ ]:
RUN_STRESS_TEST = False  # @param {type:"boolean"}
if RUN_STRESS_TEST:
    stress_path = OUTPUT_DIR / 'validation_execution_stress.json'
    stress_runner = GPU_SETUP + "\nfrom ai_trader.grpo.backtest import main\nraise SystemExit(main())\n"
    subprocess.run([sys.executable, '-u', '-c', stress_runner,
                    '--policy', str(OUTPUT_DIR / 'checkpoints/checkpoint_best.pt'),
                    '--extracted-dir', str(LOCAL_DATA_DIR), '--partition', 'validation',
                    '--episodes', str(CONFIG['evaluation_episodes']), '--seed', str(CONFIG['evaluation_seed']),
                    '--device', 'cuda', '--order-latency-ms', '250', '--spread-bps', '20',
                    '--slippage-bps', '5', '--output', str(stress_path)],
                   cwd=REPO_PATH, env=child_env, check=True)
    display(json.loads(stress_path.read_text(encoding='utf-8')))